In [ ]:
!pip install yfinance -q

import yfinance as yf
import pandas as pd
import warnings
from datetime import datetime, timedelta

# Suppress warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# ==========================================
# 1. SETTINGS
# ==========================================
LOOKBACK_DAYS = 80
TOP_PCT       = 25
BTM_PCT       = 25

MIN_PRICE     = 25      
MAX_PRICE     = 500     

# SPECIAL MAPPING FOR YAHOO FINANCE
TICKER_MAP = {
    "SPX": "^GSPC", 
    "XSP": "^XSP", 
    "DJI": "^DJI", 
    "IXIC": "^IXIC", 
    "BRK.B": "BRK-B", 
    "BF.B": "BF-B"
}

# (Your RAW_LIST remains unchanged)
RAW_LIST = """
A
AA
AAL
AAP
AAPL
ABBV
ABNB
ABT
ACN
ADBE
ADI
ADM
ADP
ADSK
AEP
AES
AFL
AFRM
AGNC
AI
AIG
AKAM
ALB
ALK
ALL
AMAT
AMD
AMGN
AMRN
AMZN
APA
APH
APTV
AVGO
AXP
BA
BABA
BAC
BAX
BBY
BEN
BIDU
BIIB
BITO
BK
BKR
BMY
BP
BSX
BX
BYND
C
CAH
CAT
CB
CCI
CCJ
CCL
CF
CFG
CHWY
CI
CL
CLF
CLX
CMCSA
CME
CNC
CNP
COF
COIN
COP
COST
CPB
CPRI
CRM
CRON
CRWD
CSCO
CSX
CTVA
CVNA
CVS
CVX
CZR
D
DAL
DD
DE
DELL
DHI
DHR
DIA
DIS
DKNG
DLR
DOCU
DOW
DRI
DVN
DXC
EA
EBAY
ED
EEM
EFA
EIX
EL
EMR
ENPH
EOG
EQR
EQT
ETSY
EVRG
EW
EWJ
EWW
EWY
EWZ
EXC
EXPE
F
FANG
FAST
FCX
FDX
FE
FI
FITB
FOXA
FSLR
FTI
FTV
FXE
FXI
GD
GDX
GE
GILD
GLD
GLW
GM
GME
GOOG
GOOGL
GPRO
GPS
GS
HAL
HBAN
HBI
HCA
HD
HIG
HLT
HOG
HOLX
HON
HPE
HPQ
HYG
IBB
IBIT
IBM
ICE
IEF
INTC
IP
IPG
IRM
IVZ
IWM
IYR
JCI
JD
JETS
JNJ
JNPR
JPM
K
KHC
KMI
KO
KR
KRE
KSS
KWEB
LEN
LI
LKQ
LLY
LNC
LOW
LQD
LUMN
LUV
LVS
LYB
LYFT
MA
MAR
MARA
MAS
MCD
MCHP
MDLZ
MDT
MET
META
MGM
MMM
MO
MOS
MPC
MRK
MRNA
MRVL
MS
MSFT
MSTR
MTB
MTCH
MU
NCLH
NEE
NEM
NET
NFLX
NKE
NOW
NRG
NTAP
NTES
NVAX
NVDA
NWL
NWSA
O
OIH
OKE
OMC
ON
ORCL
OXY
PARA
PBR
PDD
PENN
PEP
PFE
PFG
PG
PGR
PINS
PLTR
PNC
PPL
PRU
PSX
PTON
PYPL
QCOM
QQQ
RBLX
RCL
RF
RIG
RIOT
RIVN
ROKU
ROST
RTX
RUN
SBUX
SCHD
SCHW
SEDG
SHOP
SIRI
SLB
SLV
SMCI
SMH
SNAP
SNOW
SO
SOFI
SOXL
SOXS
SPG
SPX
SPXL
SPXS
SPY
SQQQ
STX
SWKS
SYF
SYY
T
TAP
TBT
TCOM
TDOC
TFC
TGT
TJX
TLT
TMO
TMUS
TPR
TQQQ
TRIP
TRV
TSLA
TSM
TSN
TT
TTD
TTWO
TXN
U
UA
UAA
UAL
UBER
ULTA
UNG
UNH
UNP
UPS
UPST
URBN
USB
USO
UVXY
V
VALE
VFC
VGK
VTR
VXX
VZ
WAB
WBA
WBD
WDC
WFC
WM
WMB
WMT
WU
WY
WYNN
X
XBI
XEL
XHB
XLB
XLC
XLE
XLF
XLI
XLK
XLP
XLRE
XLU
XLV
XLY
XOM
XOP
XRT
XRX
XSP
XYZ
YELP
YINN
ZION
ZM
"""

# ==========================================
# 2. DATA PROCESSING
# ==========================================
def get_clean_tickers():
    raw_tickers = [t.strip() for t in RAW_LIST.split('\n') if t.strip()]
    return list(set(raw_tickers))

def get_yahoo_data(tickers):
    print(f"   [YAHOO] Downloading {len(tickers)} symbols...")
    
    start_date = datetime.now() - timedelta(days=150)
    
    yahoo_tickers = []
    mapping = {}
    
    for t in tickers:
        y_t = TICKER_MAP.get(t, t.replace(".", "-"))
        yahoo_tickers.append(y_t)
        mapping[y_t] = t

    df = yf.download(yahoo_tickers, start=start_date, progress=False)
    
    closes = df['Close'] if 'Close' in df.columns else df
    closes = closes.rename(columns=mapping)
    closes = closes.sort_index()
    
    return closes

# ==========================================
# 3. MAIN SCANNER
# ==========================================
def run_scanner():
    print("--- STEP 1: PROCESSING TICKER LIST ---")
    original_tickers = get_clean_tickers()
    print(f"   Loaded {len(original_tickers)} unique symbols.")
    
    print("\n--- STEP 2: MARKET DATA (YAHOO) ---")
    data = get_yahoo_data(original_tickers)
    
    print(f"\n--- STEP 3: ANALYZING ZONES (Price filter: ${MIN_PRICE} - ${MAX_PRICE}) ---")
    
    recent_data = data.tail(LOOKBACK_DAYS)
    
    premium_list = []
    discount_list = []
    filtered_out_count = 0
    
    for col in recent_data.columns:
        try:
            series = recent_data[col].dropna()
            if len(series) < 20: 
                continue
            
            curr = float(series.iloc[-1])
            
            # Price filter
            if curr < MIN_PRICE or curr > MAX_PRICE:
                filtered_out_count += 1
                continue
            
            hi   = float(series.max())
            lo   = float(series.min())
            rng  = hi - lo
            
            if rng == 0: 
                continue
                
            loc = (curr - lo) / rng
            
            item = {'ticker': col, 'loc': round(loc*100, 1), 'price': round(curr, 2)}
            
            if loc >= (1.0 - (TOP_PCT/100)):
                premium_list.append(item)
            elif loc <= (BTM_PCT/100):
                discount_list.append(item)
                
        except Exception:
            continue

    print(f"   Skipped {filtered_out_count} assets outside price range ${MIN_PRICE}-${MAX_PRICE}.")

    # === NO MORE LIMIT - SHOW ALL QUALIFYING ASSETS ===
    discount_list.sort(key=lambda x: x['loc'])
    final_discount = discount_list   # All discounts
    
    premium_list.sort(key=lambda x: x['loc'], reverse=True)
    final_premium = premium_list     # All premiums

    # ==========================================
    # OUTPUT
    # ==========================================
    print("\n" + "="*80)
    print(f"MOST DISCOUNTED — ALL QUALIFYING ASSETS ({len(final_discount)} total)")
    print("="*80)
    for s in final_discount: 
        print(f"{s['ticker']:<8} {s['loc']:<6}%   ${s['price']:>7.2f}")

    print("\n" + "="*80)
    print(f"MOST PREMIUM — ALL QUALIFYING ASSETS ({len(final_premium)} total)")
    print("="*80)
    for s in final_premium:
        print(f"{s['ticker']:<8} {s['loc']:<6}%   ${s['price']:>7.2f}")

    # TradingView Import Strings
    print("\n\n" + "#"*80)
    print("TRADINGVIEW IMPORT STRINGS (All qualifying assets — no limit)")
    print("#"*80)
    
    print(f"\n>>> DISCOUNT LIST ({len(final_discount)} assets):")
    print(", ".join([x['ticker'] for x in final_discount]))
    
    print(f"\n>>> PREMIUM LIST ({len(final_premium)} assets):")
    print(", ".join([x['ticker'] for x in final_premium]))

run_scanner()